# OmniVoice Colab Deploy (Backend + Frontend)
Notebook này chạy đủ cả backend FastAPI và frontend Vite để test end-to-end trên Colab.

## Flow
1. Cài system deps + Python deps
2. Cài Node.js + frontend deps
3. Set env (`OMNIVOICE_MAX_CONCURRENT_LONG_JOBS`)
4. Chạy backend (port 8000)
5. Chạy frontend (port 5173)
6. Expose frontend qua ngrok


In [ ]:
!nvidia-smi || true
!apt-get -y update
!apt-get -y install ffmpeg curl
!pip install -U pip setuptools wheel

In [ ]:
import os

# TODO: đổi sang repo thật của bạn
REPO_URL = "https://github.com/your-org/OmniVoice-Space.git"
REPO_DIR = "/content/OmniVoice-Space"

if not os.path.exists(REPO_DIR):
    !git clone $REPO_URL $REPO_DIR
else:
    print("Repo đã tồn tại, dùng lại thư mục cũ")

%cd /content/OmniVoice-Space
!git pull --rebase || true

In [ ]:
%cd /content/OmniVoice-Space
!pip install -r requirements.txt
!pip install fastapi uvicorn pyngrok nest_asyncio python-multipart

In [ ]:
# Cài Node.js 20 cho frontend Vite
!curl -fsSL https://deb.nodesource.com/setup_20.x | bash -
!apt-get install -y nodejs
!node -v
!npm -v

In [ ]:
%cd /content/OmniVoice-Space/frontend
!npm ci

In [ ]:
import os
import torch

os.environ['OMNIVOICE_DEVICE'] = 'cuda' if torch.cuda.is_available() else 'cpu'
os.environ['OMNIVOICE_DTYPE'] = 'auto'
os.environ['OMNIVOICE_MAX_CONCURRENT_LONG_JOBS'] = '2'

print('OMNIVOICE_DEVICE =', os.environ['OMNIVOICE_DEVICE'])
print('OMNIVOICE_DTYPE =', os.environ['OMNIVOICE_DTYPE'])
print('OMNIVOICE_MAX_CONCURRENT_LONG_JOBS =', os.environ['OMNIVOICE_MAX_CONCURRENT_LONG_JOBS'])

In [ ]:
%cd /content/OmniVoice-Space
import subprocess, sys, time, requests

backend_cmd = [sys.executable, '-m', 'uvicorn', 'server.api:app', '--host', '0.0.0.0', '--port', '8000']
backend_proc = subprocess.Popen(backend_cmd)
print('Backend PID:', backend_proc.pid)

time.sleep(4)
print('Backend health:', requests.get('http://127.0.0.1:8000/api/health', timeout=30).json())

In [ ]:
%cd /content/OmniVoice-Space/frontend
import subprocess, time, requests

# Chạy Vite frontend, vẫn dùng proxy /api -> 127.0.0.1:8000
frontend_cmd = ['npm', 'run', 'dev', '--', '--host', '0.0.0.0', '--port', '5173']
frontend_proc = subprocess.Popen(frontend_cmd)
print('Frontend PID:', frontend_proc.pid)

time.sleep(6)
print('Frontend check status:', requests.get('http://127.0.0.1:5173', timeout=30).status_code)

In [ ]:
from pyngrok import ngrok

# Nếu cần token riêng:
# ngrok.set_auth_token('YOUR_NGROK_AUTH_TOKEN')

frontend_url = ngrok.connect(5173, bind_tls=True).public_url
backend_url = ngrok.connect(8000, bind_tls=True).public_url

print('Frontend URL:', frontend_url)
print('Backend URL :', backend_url)
print('Mở Frontend URL để test UI end-to-end')

## Gợi ý test
- Mở `Frontend URL` để dùng giao diện web
- Trong lúc chạy, kiểm tra backend: `GET {backend_url}/api/health`
- Theo dõi queue: `GET {backend_url}/api/jobs`


In [ ]:
# Dừng toàn bộ process khi test xong
for p, name in [(frontend_proc, 'frontend'), (backend_proc, 'backend')]:
    try:
        p.terminate()
        p.wait(timeout=10)
        print(f'Đã dừng {name}')
    except Exception as e:
        print(f'Không dừng được {name}:', e)

try:
    ngrok.kill()
except Exception:
    pass